In [ ]:
#import os
#import sys
#print(os.getcwd())
#current_dir = os.getcwd()
#project_root = os.path.abspath(os.path.join(current_dir,"..","..",".."))
#print(project_root)

#sys.path.append(project_root)

c:\Databricks_CLI_Demo\DAB_PROJECT\citibike_etl\notebooks\02_silver


In [ ]:
#from src.citibike.citibike_utils import get_trip_duration_mins
from citibike.citibike_utils import get_trip_duration_mins
#from src.utils.datetime_utils import timestamp_to_date_col
from utils.datetime_utils import timestamp_to_date_col
from pyspark.sql.functions import create_map, lit

In [ ]:
pipeline_id = dbutils.widgets.get("pipeline_id")
run_id = dbutils.widgets.get("run_id")
task_id = dbutils.widgets.get("task_id")
processed_timestamp = dbutils.widgets.get("processed_timestamp")
catalog = dbutils.widgets.get("catalog")

In [ ]:
df = spark.read.table(f"{catalog}.01_bronze.jc_citibike")

In [3]:
df.show()

+----------------+-------------+--------------------+--------------------+------------------+----------------+--------------------+--------------+---------+---------+-------+-------+-------------+--------------------+
|         ride_id|rideable_type|          started_at|            ended_at|start_station_name|start_station_id|    end_station_name|end_station_id|start_lat|start_lng|end_lat|end_lng|member_casual|            metadata|
+----------------+-------------+--------------------+--------------------+------------------+----------------+--------------------+--------------+---------+---------+-------+-------+-------------+--------------------+
|29DAF43DD84B4B7A|electric_bike|2025-03-20 18:58:...|2025-03-20 19:00:...|   6 St & Grand St|           HB302|Mama Johnson Fiel...|         HB404|       41|      -74|     41|    -74|       member|{pipeline_id -> p...|
|B11B4220F7195025|electric_bike|2025-03-29 11:01:...|2025-03-29 11:11:...|  Heights Elevator|           JC059|        Jersey & 3

In [4]:
df = get_trip_duration_mins(spark, df, "started_at", "ended_at", "trip_duration_mins")

In [5]:
df = timestamp_to_date_col(spark, df, "started_at", "trip_start_date")

In [6]:
df.show()

+----------------+-------------+--------------------+--------------------+------------------+----------------+--------------------+--------------+---------+---------+-------+-------+-------------+--------------------+------------------+---------------+
|         ride_id|rideable_type|          started_at|            ended_at|start_station_name|start_station_id|    end_station_name|end_station_id|start_lat|start_lng|end_lat|end_lng|member_casual|            metadata|trip_duration_mins|trip_start_date|
+----------------+-------------+--------------------+--------------------+------------------+----------------+--------------------+--------------+---------+---------+-------+-------+-------------+--------------------+------------------+---------------+
|29DAF43DD84B4B7A|electric_bike|2025-03-20 18:58:...|2025-03-20 19:00:...|   6 St & Grand St|           HB302|Mama Johnson Fiel...|         HB404|       41|      -74|     41|    -74|       member|{pipeline_id -> p...|              2.25|     

In [ ]:
df = df.withColumn("metadata", 
              create_map(
                  lit("pipeline_id"), lit(pipeline_id),
                  lit("run_id"), lit(run_id),
                  lit("task_id"), lit(task_id),
                  lit("processed_timestamp"), lit(processed_timestamp)
                  ))

In [8]:
df = df.select(
    "ride_id",
    "trip_start_date",
    "started_at",
    "ended_at",
    "start_station_name",
    "end_station_name",
    "trip_duration_mins",
    "metadata"
    )

In [9]:
df.show()

+----------------+---------------+--------------------+--------------------+------------------+--------------------+------------------+--------------------+
|         ride_id|trip_start_date|          started_at|            ended_at|start_station_name|    end_station_name|trip_duration_mins|            metadata|
+----------------+---------------+--------------------+--------------------+------------------+--------------------+------------------+--------------------+
|29DAF43DD84B4B7A|     2025-03-20|2025-03-20 18:58:...|2025-03-20 19:00:...|   6 St & Grand St|Mama Johnson Fiel...|              2.25|{pipeline_id -> p...|
|B11B4220F7195025|     2025-03-29|2025-03-29 11:01:...|2025-03-29 11:11:...|  Heights Elevator|        Jersey & 3rd| 9.733333333333333|{pipeline_id -> p...|
|18D5B30305F602B9|     2025-03-01|2025-03-01 16:05:...|2025-03-01 16:07:...|      Jersey & 3rd|       Hamilton Park| 2.183333333333333|{pipeline_id -> p...|
|532EB2D9DB68567D|     2025-03-21|2025-03-21 18:44:...|202

In [ ]:
df.write.\
    mode("overwrite").\
    option("overwriteSchema", "true").\
    saveAsTable(f"{catalog}.02_silver.jc_citibike")